# Anomaly Detection

The goal of anomaly detection is to identify activities that deviate from normal behavior and to determine the time window in which the anomaly occurs.

Anomaly detection can be viewed as a **coarse-level video understanding task** because it only distinguishes between **normal** and **abnormal** events. Its purpose is to filter out anomalous activities from regular patterns.

After an anomaly has been detected, a **classification** system can be applied to determine the specific type of activity (e.g., accident, theft, fighting, or falling).

**Pipeline**

1. Detect whether an event is normal or anomalous.
2. Locate the time interval of the anomaly.
3. Classify the anomaly into a specific activity category.

**Example**

- Anomaly Detection: "Something unusual happened between 10:05 and 10:07."
- Classification: "The unusual event was a fight."

## Multiple Instance Problem in Video Anomaly Detection

In weakly supervised anomaly detection, videos are labeled only at the video level rather than at the frame level.

A video is treated as a **bag** containing multiple temporal segments (**instances**).

- Normal video: all instances are normal.
- Anomalous video: at least one instance is anomalous.

The exact anomalous segment is unknown during training, making the task a **Multiple Instance Learning (MIL)** problem.

MIL allows models to learn anomaly localization using only video-level labels, avoiding the expensive process of frame-level annotation.

## Anomaly Ranking Model

Using weakly labeled training videos, the model learns an **anomaly ranking function** that assigns higher anomaly scores to anomalous video segments than to normal ones.

During inference, an untrimmed video is divided into multiple segments. Each segment is passed through the network, which outputs an anomaly score indicating how abnormal that segment is. Segments with high anomaly scores are identified as potential anomalies.

## Main Contributions

1. Proposed a **Multiple Instance Learning (MIL)** approach for video anomaly detection using only weakly labeled videos (normal/anomalous video-level labels) instead of costly segment-level annotations.

2. Introduced a **MIL ranking loss** with **sparsity** and **smoothness** constraints to learn anomaly scores for video segments and localize anomalies without frame-level annotations.

## Architecture

<div>
    <img src='../images/AnomalyArch.png' width="1000">
</div>

## Method

The proposed approach (summarized in Figure 1) begins with dividing surveillance videos into a ﬁxed number of segments during training. These segments make instances in a bag. Using both positive (anomalous) and negative (normal) bags, we train the anomaly detection model using the proposed deep MIL ranking loss.

### Multiple Instance Learning

In the context of supervised anomaly detection, a classifier needs temporal annotations of each segment in videos. However, obtaining temporal
annotations for videos is time consuming and laborious. In MIL, precise temporal locations of anomalous events in videos are unknown. Instead, only
video-level labels indicating the presence of an anomaly in the whole video is needed. 

A video containing anomalies is labeled as positive and a video without any anomaly is labeled as negative. Then, we represent a positive video as
a positive bag $B_a$, where different temporal segments make individual instances in the bag, $(p_1, p_2, . . . , p_m)$, where $m$ is the number of instances in the bag. We assume that **at least one of these instances contains the anomaly**.

Similarly, the negative video is denoted by a negative bag, $B_n$, where temporal segments in this bag form negative instances $(n_1, n_2, . . . , n_m)$. In the **negative bag, none of the instances contain an anomaly.**

Since the exact information (i.e. instance-level label) of the positive instances is unknown, one can optimize the objective function with respect to the maximum scored instance in each bag (assuming the responsible for the bag label is the instance with the highest score):

Hence we pick the instance with the highest score:

The highest-scoring instance within a bag is defined as:

$$
\max_{i \in B_j} \left( w \cdot \phi(x_i) - b \right)
$$

The complete MIL objective function is given by:

$$
\min_{w}
\frac{1}{z}
\sum_{j=1}^{z}
\max \left(
0,\,
1 - Y_{B_j}
\left(
\max_{i \in B_j}
\left( w \cdot \phi(x_i) - b \right)
\right)
\right)
+ \lVert w \rVert^2
$$

where $Y_{B_j}$ denotes bag-level label, $z$ is the total number of bags, and all the other variables are the same as in Hinge loss equation.

### Deep MIL Ranking Model

Anomalous behavior is difficult to define accurately , since it is quite subjective and can vary largely from person to person. Further, it is not obvious how to assign 1/0 labels to anomalies. Moreover, due to the unavailability of sufficient examples of anomaly, **anomaly detection is usually treated as low-likelihood pattern detection instead of classification problem**.

**Low-likelihood patterns** are observations or behaviors that have a low probability of occurring under the distribution of normal data, commonly denoted as $P(x)$. In anomaly detection, events that frequently occur are associated with higher probability values, while rare or unusual events have lower probabilities. Therefore, patterns with low $P(x)$ are often considered potential anomalies because they deviate from the expected behavior learned from the data.

For example, in a surveillance system monitoring a pedestrian area, people walking normally would correspond to a high-probability event (high $P(x)$) because it occurs frequently. In contrast, a vehicle entering the pedestrian zone would represent a low-probability event (low $P(x)$), making it a potential anomaly due to its rarity and deviation from normal scene activity.